# 04 - Detailed EDA on training data only

**Guardrail:** this notebook reads only `03_train.csv.gz`. Validation
and test files are not opened. The findings decide the feature and
first-model strategy used by the next notebooks.

**Outputs:** saved charts, profile tables, a findings summary, and a
machine-readable feature decision file.

In [1]:
from pathlib import Path
import json
import os

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks": ROOT = ROOT.parent
ARTIFACT_DIR = ROOT / "artifacts"
CHART_DIR = ARTIFACT_DIR / "eda" / "charts"
CHART_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / "tmp" / "task2_runtime" / "matplotlib"))

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid", context="notebook")
train_path = ARTIFACT_DIR / "03_train.csv.gz"
assert train_path.exists(), "Run Notebook 03 first."
train = pd.read_csv(train_path, low_memory=False)

DATE_COLUMNS = [
    "order_purchase_timestamp", "order_approved_at",
    "order_delivered_carrier_date", "order_delivered_customer_date",
    "order_estimated_delivery_date", "shipping_limit_date_max",
]
for column in DATE_COLUMNS:
    if column in train:
        train[column] = pd.to_datetime(train[column], errors="coerce")

print(f"Training shape: {train.shape}")
print(f"Memory: {train.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"Date range: {train['order_purchase_timestamp'].min()} to {train['order_purchase_timestamp'].max()}")

Training shape: (67529, 44)
Memory: 61.5 MB
Date range: 2016-09-15 12:16:38 to 2018-04-15 20:12:35


## Types, missing values, and memory

In [2]:
type_rows = []
for column in train.columns:
    if column in DATE_COLUMNS:
        role = "date"
    elif column.endswith("_id") or column in {"order_id", "customer_unique_id"}:
        role = "identifier"
    elif pd.api.types.is_numeric_dtype(train[column]):
        role = "numeric"
    else:
        role = "categorical/text"
    type_rows.append({
        "column": column,
        "dtype": str(train[column].dtype),
        "role": role,
        "non_null": int(train[column].notna().sum()),
        "unique": int(train[column].nunique(dropna=True)),
        "memory_bytes": int(train[column].memory_usage(deep=True)),
    })
data_profile = pd.DataFrame(type_rows)
display(data_profile.groupby("role").agg(columns=("column", "size"), memory_bytes=("memory_bytes", "sum")))

missing = pd.DataFrame({
    "column": train.columns,
    "missing_count": train.isna().sum().values,
    "missing_pct": (100 * train.isna().mean()).values,
}).sort_values(["missing_pct", "column"], ascending=[False, True])
display(missing.head(15))

data_profile.to_csv(ARTIFACT_DIR / "04_data_profile.csv", index=False)
missing.to_csv(ARTIFACT_DIR / "04_missing_values.csv", index=False)

,columns,memory_bytes
role,,
categorical/text,7,26692123
date,6,3242184
identifier,3,16409943
numeric,28,15130192


,column,missing_count,missing_pct
20,product_category_mode,1201,1.778495
22,product_description_length_mean,1197,1.772572
21,product_name_length_mean,1197,1.772572
23,product_photos_qty_mean,1197,1.772572
40,customer_seller_distance_km,344,0.509411
36,customer_geo_observations,181,0.268033
34,customer_lat,181,0.268033
35,customer_lng,181,0.268033
39,seller_geo_observations,164,0.242859
37,seller_lat,164,0.242859


## Numerical distributions, skew, outliers, and sensible ranges

In [3]:
excluded_numeric = {"late", "delivery_delay_days", "customer_zip_code_prefix", "seller_zip_code_prefix_mode"}
numeric_columns = [
    c for c in train.select_dtypes(include=np.number).columns
    if c not in excluded_numeric
]
numerical_summary = train[numeric_columns].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).T
numerical_summary["skew"] = train[numeric_columns].skew(numeric_only=True)
numerical_summary["missing_pct"] = 100 * train[numeric_columns].isna().mean()
numerical_summary = numerical_summary.reset_index(names="column")
display(numerical_summary.sort_values("skew", key=lambda s: s.abs(), ascending=False).head(12))

outlier_rows = []
for column in numeric_columns:
    series = train[column].dropna()
    if series.empty:
        continue
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    outlier_rows.append({
        "column": column,
        "lower_fence": float(lower),
        "upper_fence": float(upper),
        "outlier_count": int(((series < lower) | (series > upper)).sum()),
        "outlier_pct": float(100 * ((series < lower) | (series > upper)).mean()),
        "min": float(series.min()),
        "max": float(series.max()),
    })
outliers = pd.DataFrame(outlier_rows).sort_values("outlier_pct", ascending=False)
display(outliers.head(10))
numerical_summary.to_csv(ARTIFACT_DIR / "04_numerical_summary.csv", index=False)
outliers.to_csv(ARTIFACT_DIR / "04_outlier_summary.csv", index=False)

,column,count,mean,std,min,1%,25%,50%,75%,99%,max,skew,missing_pct
13,payment_count,67528.0,1.047625,0.397223,1.000000,1.00,1.00,1.00,1.0,2.0000,26.00,22.419509,0.001481
3,total_price,67529.0,135.888797,205.194019,2.290000,11.99,45.90,85.00,149.9,987.0000,13440.00,10.718895,0.000000
2,seller_count,67529.0,1.012054,0.114815,1.000000,1.00,1.00,1.00,1.0,2.0000,5.00,10.524938,0.000000
14,payment_value_total,67528.0,158.173637,214.285757,10.070000,22.95,61.63,104.19,175.5,1039.8568,13664.08,10.149318,0.001481
4,total_freight,67529.0,22.245936,20.002594,0.000000,7.78,14.10,16.79,23.7,98.1716,1002.29,8.581743,0.000000
0,item_count,67529.0,1.142072,0.541640,1.000000,1.00,1.00,1.00,1.0,3.0000,21.00,8.107162,0.000000
5,avg_item_price,67529.0,124.239070,185.659661,1.514286,10.99,41.00,78.97,139.9,899.0000,6735.00,7.802465,0.000000
6,max_item_price,67529.0,124.847354,186.037538,2.290000,10.99,41.90,79.00,139.9,899.0000,6735.00,7.764836,0.000000
1,product_count,67529.0,1.037702,0.222199,1.000000,1.00,1.00,1.00,1.0,2.0000,7.00,7.677312,0.000000
16,payment_type_count,67528.0,1.024020,0.153111,1.000000,1.00,1.00,1.00,1.0,2.0000,2.00,6.217627,0.001481


,column,lower_fence,upper_fence,outlier_count,outlier_pct,min,max
17,customer_lat,-28.975554,-14.608783,11081,16.453347,-33.690972,41.146203
11,product_weight_g_mean,-2025.000000,4175.000000,9769,14.469806,2.000000,40425.000000
0,item_count,1.000000,1.000000,6758,10.007552,1.000000,21.000000
4,total_freight,-0.300000,38.100000,6630,9.818004,0.000000,1002.290000
12,product_volume_cm3_mean,-22050.000000,44430.000000,5984,8.863478,288.000000,294000.000000
20,seller_lat,-26.396612,-18.974337,5832,8.657315,-32.074657,-2.498944
23,customer_seller_distance_km,-670.568683,1699.889779,5444,8.102999,0.000000,8032.940813
3,total_price,-110.100000,305.900000,5362,7.940292,2.290000,13440.000000
14,payment_value_total,-109.175000,346.305000,5309,7.861924,10.070000,13664.080000
6,max_item_price,-105.100000,286.900000,5165,7.648566,2.290000,6735.000000


## Categorical cardinality, rare values, and label relations

In [4]:
candidate_categories = [
    "customer_state", "seller_state_mode", "product_category_mode",
    "payment_type_mode", "customer_city",
]
categorical_rows = []
for column in candidate_categories:
    if column not in train:
        continue
    counts = train[column].value_counts(dropna=False, normalize=True)
    categorical_rows.append({
        "column": column,
        "unique": int(train[column].nunique(dropna=True)),
        "top_value": str(counts.index[0]),
        "top_share": float(counts.iloc[0]),
        "rare_value_count_below_1pct": int((counts < 0.01).sum()),
        "missing_pct": float(100 * train[column].isna().mean()),
    })
categorical_profile = pd.DataFrame(categorical_rows)
display(categorical_profile)
categorical_profile.to_csv(ARTIFACT_DIR / "04_categorical_profile.csv", index=False)

def label_group(column, min_orders=100):
    result = train.groupby(column, dropna=False)["late"].agg(["size", "mean"]).reset_index()
    result.columns = [column, "orders", "late_rate"]
    return result.loc[result["orders"] >= min_orders].sort_values("late_rate", ascending=False)

state_late = label_group("customer_state", min_orders=100)
seller_state_late = label_group("seller_state_mode", min_orders=100)
category_late = label_group("product_category_mode", min_orders=100)
payment_late = label_group("payment_type_mode", min_orders=100)
display(state_late.head(10))
state_late.to_csv(ARTIFACT_DIR / "04_late_by_customer_state.csv", index=False)
seller_state_late.to_csv(ARTIFACT_DIR / "04_late_by_seller_state.csv", index=False)
category_late.to_csv(ARTIFACT_DIR / "04_late_by_product_category.csv", index=False)
payment_late.to_csv(ARTIFACT_DIR / "04_late_by_payment_type.csv", index=False)

,column,unique,top_value,top_share,rare_value_count_below_1pct,missing_pct
0,customer_state,27,SP,0.402849,14,0.000000
1,seller_state_mode,22,SP,0.707429,16,0.000000
2,product_category_mode,71,bed_bath_table,0.098180,50,1.778495
3,payment_type_mode,4,credit_card,0.763479,1,0.001481
4,customer_city,3747,sao paulo,0.146900,3738,0.000000


,customer_state,orders,late_rate
1,AL,298,0.278523
9,MA,548,0.215328
5,CE,948,0.182489
24,SE,246,0.170732
16,PI,348,0.166667
18,RJ,8965,0.165867
4,BA,2307,0.149978
13,PA,718,0.133705
7,ES,1445,0.132872
26,TO,197,0.131980


## Dates, seasonality, weekday, fixed holidays, and geography

In [5]:
analysis = train.copy()
purchase = analysis["order_purchase_timestamp"]
analysis["purchase_month"] = purchase.dt.to_period("M").astype(str)
analysis["purchase_weekday"] = purchase.dt.day_name()
analysis["purchase_is_weekend"] = purchase.dt.dayofweek.ge(5)
fixed_holiday_month_days = {
    (1, 1), (4, 21), (5, 1), (9, 7), (10, 12), (11, 2), (11, 15), (12, 25)
}
analysis["purchase_is_fixed_holiday"] = [
    (month, day) in fixed_holiday_month_days
    for month, day in zip(purchase.dt.month, purchase.dt.day)
]
analysis["estimated_delivery_lead_days"] = (
    analysis["order_estimated_delivery_date"] - purchase
).dt.total_seconds() / 86400

monthly = analysis.groupby("purchase_month")["late"].agg(orders="size", late_rate="mean").reset_index()
weekday = analysis.groupby("purchase_weekday")["late"].agg(orders="size", late_rate="mean").reset_index()
weekend = analysis.groupby("purchase_is_weekend")["late"].agg(orders="size", late_rate="mean").reset_index()
holiday = analysis.groupby("purchase_is_fixed_holiday")["late"].agg(orders="size", late_rate="mean").reset_index()
monthly.to_csv(ARTIFACT_DIR / "04_monthly_late_rate.csv", index=False)
weekday.to_csv(ARTIFACT_DIR / "04_weekday_late_rate.csv", index=False)
weekend.to_csv(ARTIFACT_DIR / "04_weekend_late_rate.csv", index=False)
holiday.to_csv(ARTIFACT_DIR / "04_fixed_holiday_late_rate.csv", index=False)
display(weekday.sort_values("late_rate", ascending=False))
display(holiday)

distance_frame = analysis.loc[analysis["customer_seller_distance_km"].notna(), ["customer_seller_distance_km", "late"]].copy()
distance_frame["distance_band"] = pd.qcut(
    distance_frame["customer_seller_distance_km"], q=5, duplicates="drop"
)
distance_late = distance_frame.groupby("distance_band", observed=True)["late"].agg(orders="size", late_rate="mean").reset_index()
distance_late["distance_band"] = distance_late["distance_band"].astype(str)
distance_late.to_csv(ARTIFACT_DIR / "04_distance_late_rate.csv", index=False)
display(distance_late)

,purchase_weekday,orders,late_rate
1,Monday,10734,0.100149
0,Friday,9884,0.094699
5,Tuesday,10729,0.092646
2,Saturday,7593,0.089688
6,Wednesday,10339,0.086082
3,Sunday,8245,0.083808
4,Thursday,10005,0.082859


,purchase_is_fixed_holiday,orders,late_rate
0,False,66668,0.090703
1,True,861,0.056911


,distance_band,orders,late_rate
0,"(-0.001, 133.694]",13437,0.050160
1,"(133.694, 354.875]",13437,0.082757
2,"(354.875, 544.051]",13437,0.090124
3,"(544.051, 886.405]",13437,0.102106
4,"(886.405, 8032.941]",13437,0.125698


## Numerical relations with the target

In [6]:
relation_columns = [
    c for c in numeric_columns + ["estimated_delivery_lead_days"]
    if c in analysis.columns
]
correlations = (
    analysis[relation_columns + ["late"]]
    .corr(numeric_only=True)["late"]
    .drop("late")
    .sort_values(key=lambda s: s.abs(), ascending=False)
    .rename_axis("column")
    .reset_index(name="correlation_with_late")
)
display(correlations.head(12))
correlations.to_csv(ARTIFACT_DIR / "04_numeric_target_correlations.csv", index=False)

,column,correlation_with_late
0,customer_seller_distance_km,0.086419
1,customer_lng,0.082651
2,customer_lat,0.061400
3,total_freight,0.046597
4,estimated_delivery_lead_days,-0.028093
5,avg_item_price,0.027407
6,max_item_price,0.026725
7,seller_count,-0.025871
8,product_weight_g_mean,0.025670
9,payment_value_total,0.024776


## Save charts

In [7]:
# 1. Class distribution
fig, ax = plt.subplots(figsize=(7, 4.5))
order = ["on_time", "late"]
sns.countplot(data=analysis, x="delivery_label", order=order, hue="delivery_label", legend=False, ax=ax, palette="Set2")
ax.set(title="Training label distribution", xlabel="Delivery label", ylabel="Orders")
fig.tight_layout(); fig.savefig(CHART_DIR / "01_class_distribution.png", dpi=160); plt.show(); plt.close(fig)

# 2. Missing values
missing_plot = missing.loc[missing["missing_count"].gt(0)].head(15).sort_values("missing_pct")
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=missing_plot, x="missing_pct", y="column", hue="column", legend=False, ax=ax, palette="viridis")
ax.set(title="Top missing-value rates (training only)", xlabel="Missing (%)", ylabel="")
fig.tight_layout(); fig.savefig(CHART_DIR / "02_missing_values.png", dpi=160); plt.show(); plt.close(fig)

# 3. Selected numerical distributions
plot_columns = [c for c in ["total_price", "total_freight", "payment_value_total", "customer_seller_distance_km", "item_count", "estimated_delivery_lead_days"] if c in analysis]
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for ax, column in zip(axes.flat, plot_columns):
    sns.histplot(analysis[column], bins=40, ax=ax, color="#6f2c91")
    ax.set_title(column)
for ax in axes.flat[len(plot_columns):]: ax.set_visible(False)
fig.suptitle("Selected training distributions", y=1.02)
fig.tight_layout(); fig.savefig(CHART_DIR / "03_numerical_distributions.png", dpi=160, bbox_inches="tight"); plt.show(); plt.close(fig)

# 4. Monthly volume and late rate
# Exclude extremely small partial months from the chart scale while
# preserving them in the saved monthly table.
monthly_plot = monthly.loc[monthly["orders"].ge(100)].copy()
fig, ax1 = plt.subplots(figsize=(11, 5))
ax1.bar(monthly_plot["purchase_month"], monthly_plot["orders"], color="#d9b3e6", label="Orders")
ax1.set_ylabel("Orders"); ax1.tick_params(axis="x", rotation=60)
ax2 = ax1.twinx(); ax2.plot(monthly_plot["purchase_month"], 100 * monthly_plot["late_rate"], color="#e76f51", marker="o", label="Late rate")
ax2.set_ylabel("Late rate (%)"); ax1.set_title("Monthly volume and late rate - training months with 100+ orders")
fig.tight_layout(); fig.savefig(CHART_DIR / "04_monthly_volume_late_rate.png", dpi=160); plt.show(); plt.close(fig)

# 5. Customer-state relation
state_plot = state_late.nlargest(12, "orders").sort_values("late_rate")
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=state_plot, x="late_rate", y="customer_state", hue="customer_state", legend=False, ax=ax, palette="magma")
ax.set(title="Late rate by high-volume customer state", xlabel="Late rate", ylabel="State")
fig.tight_layout(); fig.savefig(CHART_DIR / "05_customer_state_late_rate.png", dpi=160); plt.show(); plt.close(fig)

# 6. Geographic distance relation
fig, ax = plt.subplots(figsize=(9, 4.5))
sns.barplot(data=distance_late, x="distance_band", y="late_rate", hue="distance_band", legend=False, ax=ax, palette="crest")
ax.set(title="Late rate by customer-seller distance quintile", xlabel="Distance band (km)", ylabel="Late rate")
ax.tick_params(axis="x", rotation=25)
fig.tight_layout(); fig.savefig(CHART_DIR / "06_distance_late_rate.png", dpi=160); plt.show(); plt.close(fig)

# 7. Correlation heatmap without outcome/future columns
heat_columns = [c for c in ["late", "total_price", "total_freight", "payment_value_total", "customer_seller_distance_km", "item_count", "product_weight_g_mean", "estimated_delivery_lead_days"] if c in analysis]
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(analysis[heat_columns].corr(numeric_only=True), cmap="vlag", center=0, annot=True, fmt=".2f", ax=ax)
ax.set_title("Training-only numerical correlation matrix")
fig.tight_layout(); fig.savefig(CHART_DIR / "07_correlation_heatmap.png", dpi=160); plt.show(); plt.close(fig)

print(f"Saved {len(list(CHART_DIR.glob('*.png')))} charts to {CHART_DIR.relative_to(ROOT)}")

C:\Users\Admin\AppData\Local\Temp\ipykernel_25608\3287785078.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig(CHART_DIR / "01_class_distribution.png", dpi=160); plt.show(); plt.close(fig)


C:\Users\Admin\AppData\Local\Temp\ipykernel_25608\3287785078.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig(CHART_DIR / "02_missing_values.png", dpi=160); plt.show(); plt.close(fig)


C:\Users\Admin\AppData\Local\Temp\ipykernel_25608\3287785078.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig(CHART_DIR / "03_numerical_distributions.png", dpi=160, bbox_inches="tight"); plt.show(); plt.close(fig)


C:\Users\Admin\AppData\Local\Temp\ipykernel_25608\3287785078.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig(CHART_DIR / "04_monthly_volume_late_rate.png", dpi=160); plt.show(); plt.close(fig)


C:\Users\Admin\AppData\Local\Temp\ipykernel_25608\3287785078.py:41: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig(CHART_DIR / "05_customer_state_late_rate.png", dpi=160); plt.show(); plt.close(fig)


C:\Users\Admin\AppData\Local\Temp\ipykernel_25608\3287785078.py:48: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig(CHART_DIR / "06_distance_late_rate.png", dpi=160); plt.show(); plt.close(fig)


Saved 7 charts to artifacts\eda\charts


C:\Users\Admin\AppData\Local\Temp\ipykernel_25608\3287785078.py:55: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig(CHART_DIR / "07_correlation_heatmap.png", dpi=160); plt.show(); plt.close(fig)


## Findings and feature/model decisions

In [8]:
base_numeric = [
    "item_count", "product_count", "seller_count", "total_price",
    "total_freight", "avg_item_price", "max_item_price",
    "product_category_count", "product_name_length_mean",
    "product_description_length_mean", "product_photos_qty_mean",
    "product_weight_g_mean", "product_volume_cm3_mean",
    "payment_count", "payment_value_total", "max_payment_installments",
    "payment_type_count", "customer_seller_distance_km",
]
base_categorical = [
    "customer_state", "seller_state_mode", "product_category_mode", "payment_type_mode"
]
engineered_numeric = [
    "estimated_delivery_lead_days", "purchase_year", "purchase_month",
    "purchase_dayofweek", "purchase_hour", "purchase_is_weekend",
    "purchase_is_fixed_holiday",
]
leakage_columns = [
    "late", "delivery_label", "delivery_delay_days", "order_status",
    "order_approved_at", "order_delivered_carrier_date",
    "order_delivered_customer_date", "shipping_limit_date_max",
]
identifier_or_high_cardinality = [
    "order_id", "customer_id", "customer_unique_id", "customer_city",
    "customer_zip_code_prefix", "seller_zip_code_prefix_mode",
]
feature_decisions = {
    "base_numeric_features": [c for c in base_numeric if c in train],
    "base_categorical_features": [c for c in base_categorical if c in train],
    "engineered_numeric_features": engineered_numeric,
    "leakage_columns": leakage_columns,
    "identifier_or_high_cardinality_columns": identifier_or_high_cardinality,
    "missing_values": "median for numeric; most-frequent for categorical; train fit only",
    "categorical_encoding": "one-hot with unknown handling and rare-category grouping",
    "numeric_scaling": "standard scaling for logistic regression; train fit only",
    "first_model": "class-weighted logistic regression",
    "primary_metric": "average precision (PR-AUC)",
    "reason": "interpretable sparse baseline suited to an imbalanced binary target",
}
(ARTIFACT_DIR / "04_feature_decisions.json").write_text(
    json.dumps(feature_decisions, indent=2), encoding="utf-8"
)

late_rate = float(train["late"].mean())
imbalance_ratio = float(max(late_rate, 1-late_rate) / min(late_rate, 1-late_rate))
strongest = correlations.iloc[0]
top_missing = missing.loc[missing["missing_count"].gt(0)].head(3)
missing_text = ", ".join(
    f"{row.column} ({row.missing_pct:.1f}%)" for row in top_missing.itertuples()
) or "none"
findings = f'''# Training-only EDA findings

- The training split contains **{len(train):,} orders** from {train['order_purchase_timestamp'].min().date()} through {train['order_purchase_timestamp'].max().date()}.
- The late rate is **{late_rate:.2%}**, an approximately **{imbalance_ratio:.1f}:1** majority/minority imbalance. Accuracy alone is therefore unsuitable.
- Highest missingness among present columns: {missing_text}. Missing values will be imputed by transformers fit on training data only.
- The strongest non-outcome numerical correlation with `late` is **{strongest['column']}** ({strongest['correlation_with_late']:.3f}); relationships are not assumed to be linear from correlation alone.
- Prices, freight, payment totals, product size/weight, item counts, promised lead time, and customer-seller distance are skewed and can contain legitimate high-value orders. Median imputation plus scaling is safer than deleting outliers without business evidence.
- Customer/seller state, product category, payment type, date seasonality, weekday/weekend, fixed-holiday, and distance groups show different late rates and are retained as candidate signals.
- IDs, raw ZIP prefixes, and city are excluded to avoid memorization/high cardinality. Reviews and actual delivery/carrier/delay fields are future information and are excluded for leakage control.
- First model: a class-weighted logistic regression. Primary validation metric: PR-AUC, with late-class precision, recall, F1, ROC-AUC, and balanced accuracy reported as supporting metrics.
'''
(ARTIFACT_DIR / "04_eda_findings.md").write_text(findings, encoding="utf-8")
eda_summary = {
    "training_rows": int(len(train)),
    "training_columns": int(train.shape[1]),
    "training_late_rate": late_rate,
    "class_imbalance_ratio": imbalance_ratio,
    "saved_chart_count": len(list(CHART_DIR.glob("*.png"))),
    "validation_or_test_opened": False,
}
(ARTIFACT_DIR / "04_eda_summary.json").write_text(
    json.dumps(eda_summary, indent=2), encoding="utf-8"
)
print(findings)

# Training-only EDA findings

- The training split contains **67,529 orders** from 2016-09-15 through 2018-04-15.
- The late rate is **9.03%**, an approximately **10.1:1** majority/minority imbalance. Accuracy alone is therefore unsuitable.
- Highest missingness among present columns: product_category_mode (1.8%), product_description_length_mean (1.8%), product_name_length_mean (1.8%). Missing values will be imputed by transformers fit on training data only.
- The strongest non-outcome numerical correlation with `late` is **customer_seller_distance_km** (0.086); relationships are not assumed to be linear from correlation alone.
- Prices, freight, payment totals, product size/weight, item counts, promised lead time, and customer-seller distance are skewed and can contain legitimate high-value orders. Median imputation plus scaling is safer than deleting outliers without business evidence.
- Customer/seller state, product category, payment type, date seasonality, weekday/weekend, fixed-h